In [1]:
import numpy as np
import pandas as pd
from my_helpers import *
import csv

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Below identifiers correspond to evaluation protocol 1, 2 and 3 respectively
# identifiers - "TouchPose_crossSession", "TouchPose_crossSubjects", "TouchPose_crossGestures"
identifier = "TouchPose_crossGestures"
data_path = "../data/eth_data/"
model_path = "../models/"
new_data = "../data/27Arish"

# Information from Report

**Skeleton Data**   
The valid skeleton data we collected from leap motion controller covered 147 trails,
with an average number of frames per trail of 1316. The frame rate of Leap motion data is
30 fps. 

**Cap_img data**   
The capacity image and stylus data were recorded from Wacom tablet. The capacitive
image has the resolution 72*42. We collected 132 trails of capacity image and 201 trails
of pen data in total. The average number of frames of capacitive data per trial is 13203
frames, and the number of frames of pen data per trial is 12262. The recording rate is
about 80 fps. 

In [4]:
cap_text_cols = ['TimeStamps', 'MergeTimeStamps', 'ScanSizeX', 'ScanSizeY']
cap_numeric_cols =  list(map(str,range(3024)))
cap_cols = cap_text_cols + cap_numeric_cols
wrist_cols = ['wrist_x','wrist_y','wrist_z']
finger_cols = sum([[f'{x}_{y}_x', f'{x}_{y}_y', f'{x}_{y}_z'] for x in ['Thumb','Index','Middle','Ring','Pinky'] for y in range(5)],[]) 
skel_cols = ['Timestamp'] + wrist_cols + finger_cols
last_fingers_cols = [x for x in finger_cols if '_4_' in x]
sniffer = csv.Sniffer()

#read cap img files for 6 gestures
cap_img1 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_1_UP_100_rawdata.csv",sniffer,cap_cols)
cap_img2 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_2_DT_100_rawdata.csv",sniffer,cap_cols)
cap_img3 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_3_DQ_100_rawdata.csv",sniffer,cap_cols)
cap_img4 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_4_LQ_100_rawdata.csv",sniffer,cap_cols)
cap_img5 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_5_LT_100_rawdata.csv",sniffer,cap_cols)
cap_img6 = read_cap_img("../data/DATA3003/16Yaman/YAMAN_6_UP_100_rawdata.csv",sniffer,cap_cols)
#concat into one file
cap_img = pd.concat([cap_img1,cap_img2,cap_img3,cap_img4,cap_img5,cap_img6],ignore_index=True )
#process cap_img with truncation
cap_img = process_cap_img(cap_img,trunc=True)
skeleton = read_skeleton("../data/DATA3003/16Yaman/useryaman_2023-03-30_11-35-18_1644_handleap.csv",sniffer,skel_cols)
#process skeleton with truncation
skeleton = process_skeleton(skeleton,last_fingers_cols,trunc=True)

In [5]:
print(cap_img.dtypes)
skeleton.dtypes

Timestamp          float64
MergeTimeStamps    float64
ScanSizeX          float64
ScanSizeY            int64
cap_img             object
dtype: object


Timestamp    float64
skeleton      object
dtype: object

In [7]:
def process_skeleton2(skeleton,last_fingers_cols,trunc=False):
    #we keep only four first measurements for each finger instead of 5
    if trunc : 
        skeleton=skeleton.drop(last_fingers_cols,axis=1)
    hand_cords = skeleton.drop("Timestamp",axis=1)
    skeleton['skeleton'] = list(hand_cords.values.reshape(-1,21,3).astype(int))
    skeleton = skeleton.drop(hand_cords.columns, axis=1)
    skeleton[skeleton.columns.drop('skeleton')] = skeleton[skeleton.columns.drop('skeleton')].apply(pd.to_numeric, errors='coerce')
    skeleton.dropna(inplace=True)
    skeleton.Timestamp = skeleton.Timestamp*1000
    return skeleton

skeleton = read_skeleton("../data/DATA3003/16Yaman/useryaman_2023-03-30_11-35-18_1644_handleap.csv",sniffer,skel_cols)
#process skeleton with truncation
skeleton = process_skeleton2(skeleton,last_fingers_cols,trunc=True)
display(skeleton)

,Timestamp,skeleton
0,1.680168e+12,"[[-101, 239, 167], [-65, 245, 160], [-65, 245,..."
1,1.680168e+12,"[[-96, 238, 164], [-61, 244, 157], [-61, 244, ..."
2,1.680168e+12,"[[-90, 237, 163], [-55, 243, 156], [-55, 243, ..."
3,1.680168e+12,"[[-83, 237, 162], [-48, 243, 155], [-48, 243, ..."
4,1.680168e+12,"[[-79, 237, 161], [-44, 244, 154], [-44, 244, ..."
...,...,...
1639,1.680169e+12,"[[-162, 278, 137], [-143, 260, 119], [-143, 26..."
1640,1.680169e+12,"[[-167, 274, 135], [-148, 255, 117], [-148, 25..."
1641,1.680169e+12,"[[-177, 260, 137], [-158, 242, 118], [-158, 24..."
1642,1.680169e+12,"[[-182, 239, 141], [-159, 223, 126], [-159, 22..."


In [84]:
display(skeleton)
cap_img

,Timestamp,skeleton
0,1.680168e+12,"[[-101.15926361083984, 239.8048095703125, 167...."
1,1.680168e+12,"[[-96.8387680053711, 238.14573669433597, 164.5..."
2,1.680168e+12,"[[-90.62974548339844, 237.73812866210935, 163...."
3,1.680168e+12,"[[-83.21353149414062, 237.57627868652344, 162...."
4,1.680168e+12,"[[-79.19513702392578, 237.2896881103516, 161.6..."
...,...,...
1639,1.680169e+12,"[[-162.6908416748047, 278.8058166503906, 137.2..."
1640,1.680169e+12,"[[-167.2104034423828, 274.0877990722656, 135.8..."
1641,1.680169e+12,"[[-177.12181091308594, 260.3567810058594, 137...."
1642,1.680169e+12,"[[-182.610595703125, 239.5504150390625, 141.61..."


,Timestamp,MergeTimeStamps,ScanSizeX,ScanSizeY,cap_img
0,1.680168e+12,1.680168e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,1.680168e+12,1.680168e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,1.680168e+12,1.680168e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,1.680168e+12,1.680168e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,1.680168e+12,1.680168e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
...,...,...,...,...,...
24584,1.680169e+12,1.680169e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
24585,1.680169e+12,1.680169e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
24586,1.680169e+12,1.680169e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
24587,1.680169e+12,1.680169e+12,72.0,41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [85]:
skeleton.Timestamp.diff().mean()

722.9634460709259

In [86]:
cap_img.Timestamp.diff().mean()

48.754951803798754

There are some large jumps in data. We can't adress it now as we will have discountinous bins. We will remove the affected capacitive images after assignment.

It looks like MergeTimeStamps is the biggest smaller value mutiple of 5 with respect to TimeStamps.

In [96]:
# we will make both df overlap
max_cap, min_cap = max(cap_img.Timestamp), min(cap_img.Timestamp)
max_skel, min_skel = max(skeleton.Timestamp), min(skeleton.Timestamp)
print(max_skel > max_cap)
print(min_skel < min_cap)

False
True


Get overlapping between skeleton and capacitive images

In [99]:
mask_skel = (skeleton.Timestamp >= min_cap) & (skeleton.Timestamp <= max_cap)
mask_skel[mask_skel]

174     True
175     True
176     True
177     True
178     True
        ... 
1639    True
1640    True
1641    True
1642    True
1643    True
Name: Timestamp, Length: 1470, dtype: bool

Hence we will also take rows 173 and 1644 if possible

In [100]:
loc1=mask_skel[mask_skel].first_valid_index()
locn=mask_skel[mask_skel].last_valid_index()
if loc1>0:
    mask_skel.loc[loc1-1]=True
if locn<len(skeleton)-1:
    mask_skel.loc[locn+1]=True

In [101]:
mask_skel[mask_skel==True]

173     True
174     True
175     True
176     True
177     True
        ... 
1639    True
1640    True
1641    True
1642    True
1643    True
Name: Timestamp, Length: 1471, dtype: bool

In [102]:
skeleton=skeleton[mask_skel].reset_index(drop=True)
len(skeleton)

1471

In [103]:
len(cap_img.cap_img)

24588

Hence we want to get 1471 observations out of 1644, we will map skeleton to average of capacitive images.

In [49]:
bins = skeleton.Timestamp
x = cap_img.Timestamp
cap_img['where_to'] = np.digitize(x, bins)
cap_img.where_to=cap_img.where_to.map(lambda x: x-1)
cap_img.where_to.unique()

array([   0,   54,   72,   73,   74,   75,   76,   77,   78,   79,   80,
         81,   82,   83,   84,   85,   86,   87,   88,   89,   90,   91,
        112,  115,  116,  117,  118,  119,  120,  121,  149,  150,  151,
        152,  156,  157,  158,  159,  164,  165,  166,  168,  169,  170,
        171,  172,  173,  174,  175,  176,  177,  178,  179,  180,  181,
        182,  183,  184,  185,  205,  242,  243,  244,  245,  246,  247,
        248,  249,  250,  269,  270,  271,  272,  276,  277,  278,  286,
        287,  288,  329,  452,  453,  470,  471,  472,  473,  474,  475,
        476,  477,  478,  479,  480,  481,  482,  510,  511,  512,  513,
        521,  565,  567,  568,  569,  571,  572,  754,  857,  858,  859,
        860,  861,  862,  863,  864,  865,  866,  867,  868,  869,  870,
        871,  872,  873,  874,  875,  876,  877,  878,  879,  881,  882,
        883,  884,  885,  886,  887,  888,  889,  890,  891,  892,  893,
        894,  895,  896,  897,  898,  899,  900,  9

check, looks in order

In [50]:
display(cap_img[(cap_img.where_to==1000)].Timestamp.iloc[0])
display(cap_img[(cap_img.where_to==1000)].Timestamp.iloc[1])
display(skeleton.iloc[1000].Timestamp)
#maybe find a way to rearrange better
display(skeleton.iloc[1001].Timestamp)

1680168795507.0

1680168795520.0

1680168795494.6748

1680168795527.2131

These are the skeleton points that present too much of a jump. We remove all capacitive images that were assigned to them

In [52]:
threshhold = 80
ranking =skeleton.Timestamp.diff().sort_values(ascending=False)
ranking=ranking[ranking > threshhold]
ranking.index=ranking.index-1
ranking

54      376691.893066
754     368034.704834
0       211933.413574
857      45180.883789
329      41485.784180
572      20275.489014
1135     16064.123047
521      14939.534912
600      14597.821533
Name: Timestamp, dtype: float64

In [53]:
cap_img.columns

Index(['Timestamp', 'MergeTimeStamps', 'ScanSizeX', 'ScanSizeY', 'cap_img',
       'where_to'],
      dtype='object')

In [54]:
cols = cap_img.columns.drop(['cap_img','where_to', 'MergeTimeStamps', 'Timestamp'])
dict = {x: 'first' for x in cols}
dict['cap_img'] = 'mean'
prev = len(cap_img)
cap_img=cap_img[~cap_img.where_to.isin(ranking.index)]
print('now', len(cap_img), 'discarded frames', prev - len(cap_img), 'out of', prev)
res=cap_img.groupby('where_to').agg(dict)
res

now 3603 discarded frames 20985 out of 24588


,ScanSizeX,ScanSizeY,cap_img
where_to,,,
72,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
73,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
74,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
75,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
76,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...
1130,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1131,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1132,72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [58]:
cap_img.columns

Index(['Timestamp', 'MergeTimeStamps', 'ScanSizeX', 'ScanSizeY', 'cap_img',
       'where_to'],
      dtype='object')

In [55]:
data=skeleton.iloc[res.index]
data = pd.concat([data,res],axis=1).reset_index(drop=True)
data

,Timestamp,skeleton,ScanSizeX,ScanSizeY,cap_img
0,1.680168e+12,"[[-244, 272, 257], [-220, 253, 240], [-220, 25...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,1.680168e+12,"[[-245, 271, 257], [-220, 253, 240], [-220, 25...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,1.680168e+12,"[[-246, 271, 257], [-222, 253, 240], [-222, 25...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,1.680168e+12,"[[-247, 270, 257], [-222, 252, 240], [-222, 25...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,1.680168e+12,"[[-247, 270, 256], [-223, 252, 239], [-223, 25...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...
371,1.680169e+12,"[[-264, 288, 333], [-248, 274, 309], [-248, 27...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
372,1.680169e+12,"[[-266, 284, 336], [-251, 270, 310], [-251, 27...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
373,1.680169e+12,"[[-268, 283, 338], [-254, 268, 312], [-254, 26...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
374,1.680169e+12,"[[-268, 281, 333], [-251, 267, 311], [-251, 26...",72.0,41,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
